### Dependencies

In [6]:
from apify_client import ApifyClient
from dotenv import load_dotenv
import pandas as pd
import re
import requests
import json
import os
import shutil
from datetime import datetime
from pathlib import Path
from apify_class import Apify

## Save Profile Picture

In [ ]:
import requests

url = "https://instagram.fisb17-1.fna.fbcdn.net/v/t51.82787-19/525781976_18071951138078352_5065639200253104830_n.jpg?efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmMyIn0&_nc_ht=instagram.fisb17-1.fna.fbcdn.net&_nc_cat=100&_nc_oc=Q6cZ2gEVYdsLSF2tU9vYFPcHw_iXoZX9-klVLnAKky0N5CuNcq-nVZXbi2OhBIAtMSffUm8&_nc_ohc=nlBRVVj9-x8Q7kNvwE5CGHZ&_nc_gid=82PAI0sd7oaW1dRclrXGhg&edm=AP4sbd4BAAAA&ccb=7-5&oh=00_AfyLNdyV5AJgXBsYec5Tov01ZXgdONbQTpzisQoqrZfykQ&oe=69C87744&_nc_sid=7a9f4b"
# Send request
response = requests.get(url)
# Check if download was successful
if response.status_code == 200:
    with open("../data/_saras.archives/_saras.archives.jpg", "wb") as f:
        f.write(response.content)

In [1]:
import psycopg2

def get_info(brand: str, influencers: list[str]):
    conn = psycopg2.connect(database="postgres", user="postgres", password=1040)
    cur = conn.cursor()

    query = """
        SELECT profile_pic, username, name, followers, following, location, businesscategoryname
        FROM influencers
        WHERE influencerid = %s
    """
    cur = conn.cursor()

    result = []

    for influencer in influencers:
        cur.execute(query, (influencer,))
        result.append(cur.fetchall())
    cur.close()
    
    return result

In [2]:
import json

with open('similarity_matches.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

influencers_for_brand = data.get('top_influencers_for_brand')
count = -1 
for brand, influencers in influencers_for_brand.items():
    count+=1
    if count <= 1:
        continue
    print(f'Brand: {brand}\nRec. Influencers: {influencers}')
    r = get_info(brand, influencers)
    for rr in r:
        print(rr)
    break

Brand: 48310883881
Rec. Influencers: ['11625030684', '2962038165', '9043701529', '63941193886', '49966487722', '5761442945', '2296689966', '530921038', '14649214091', '357579209']
[('https://upclout-profile-pics.s3.us-west-1.amazonaws.com/profile-pics/emannjafferr.jpg', 'emannjafferr', 'Eman Jaffer', 26925, 411, 'karachi', 'Digital creator')]
[('https://upclout-profile-pics.s3.us-west-1.amazonaws.com/profile-pics/hafsa._.khan1.jpg', 'hafsa._.khan1', 'Hafsa Shaheer Khan 🤍', 2949021, 88, 'karachi', 'Digital creator')]
[('https://upclout-profile-pics.s3.us-west-1.amazonaws.com/profile-pics/malaekaaa.jpg', 'malaekaaa', 'malaeka', 51667, 1473, 'islamabad', 'Personal blog')]
[('https://upclout-profile-pics.s3.us-west-1.amazonaws.com/profile-pics/shahiherself.jpg', 'shahiherself', 'Shahi 🌸', 107639, 258, 'Nan', 'None,Digital creator')]
[('https://upclout-profile-pics.s3.us-west-1.amazonaws.com/profile-pics/ezamaryamm.jpg', 'ezamaryamm', 'عزا مريم', 25618, 30, 'Nan', 'None,Digital creator')]
[

In [7]:
import boto3
import requests
import os
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

BUCKET = os.getenv('AWS_S3_BUCKET')

def upload_profile_pic(username: str, image_url: str) -> str:
    """
    Downloads profile pic from Instagram URL,
    uploads to S3, returns the public URL.
    """
    # Download the image
    response = requests.get(image_url)
    if response.status_code != 200:
        return None

    # Upload to S3
    key = f"profile-pics/{username}.jpg"
    s3.put_object(
        Bucket=BUCKET,
        Key=key,
        Body=response.content,
        ContentType='image/jpeg'
    )

    # Return the public URL
    public_url = f"https://{BUCKET}.s3.{os.getenv('AWS_REGION')}.amazonaws.com/{key}"
    return public_url

url = upload_profile_pic("learningwithmahnoor", "https://instagram.fisb17-1.fna.fbcdn.net/v/t51.2885-19/505099258_18512564437036255_9141505177023149179_n.jpg?efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmMyIn0&_nc_ht=instagram.fisb17-1.fna.fbcdn.net&_nc_cat=107&_nc_oc=Q6cZ2gE7P04F_melxoYbJsNcy0ilJiXFAZWb64GbpkV8hyqQP9_iMUU6G9DPNUDKjaEBSh8&_nc_ohc=ORhLcbAIAaMQ7kNvwEeasYK&_nc_gid=aovY5_DgmZwqwODDg-g4KA&edm=AP4sbd4BAAAA&ccb=7-5&oh=00_AfxmdHvdeX9wE_4PPjZ8EVqmTtziVPBHLpR4ry02rHfDpw&oe=69D16D0A&_nc_sid=7a9f4b")
print(url)

https://upclout-profile-pics.s3.us-west-1.amazonaws.com/profile-pics/learningwithmahnoor.jpg


In [6]:
import boto3
import os
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

BUCKET = os.getenv('AWS_S3_BUCKET')
REGION = os.getenv('AWS_REGION')

def upload_all_profile_pics(base_path='../data/influencers_DONE'):
    uploaded = 0
    failed = []

    for username in os.listdir(base_path):
        folder = os.path.join(base_path, username)
        if not os.path.isdir(folder):
            continue

        jpg_path = os.path.join(folder, f"{username}.jpg")
        if not os.path.exists(jpg_path):
            failed.append((username, "No .jpg found"))
            continue

        try:
            key = f"profile-pics/{username}.jpg"
            s3.upload_file(
                jpg_path,
                BUCKET,
                key,
                ExtraArgs={'ContentType': 'image/jpeg'}
            )
            uploaded += 1
            print(f"✅ [{uploaded}] {username}")
        except Exception as e:
            failed.append((username, str(e)))
            print(f"❌ {username} — {e}")

    print(f"\nDone! Uploaded: {uploaded} | Failed: {len(failed)}")
    if failed:
        print("Failed uploads:")
        for name, reason in failed:
            print(f"  - {name}: {reason}")

    return uploaded, failed

In [ ]:
upload_all_profile_pics()

In [ ]:
import json

def extract_titles_to_profiles(json_path='saved_posts.json', profiles_path='../insta_profiles.txt'):
    """Extract all titles from saved_posts.json and append them to insta_profiles.txt (no duplicates with existing entries)."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    titles = [post['title'] for post in data.get('saved_saved_media', []) if 'title' in post]

    # Read existing profiles to avoid duplicates
    with open(profiles_path, 'r', encoding='utf-8') as f:
        existing = set(line.strip() for line in f if line.strip())

    # Filter out titles already present
    new_titles = [t for t in titles if t not in existing]
    # Also deduplicate within the new titles while preserving order
    seen = set()
    unique_new = []
    for t in new_titles:
        if t not in seen:
            seen.add(t)
            unique_new.append(t)

    # Append to file
    with open(profiles_path, 'a', encoding='utf-8') as f:
        for title in unique_new:
            f.write(title + '\n')

    print(f"Added {len(unique_new)} new profiles (skipped {len(titles) - len(unique_new)} duplicates)")
    return unique_new

# Run it
extract_titles_to_profiles()


In [8]:
def remove_dup_influencers() -> None:

    with open("../insta_profiles.txt", "r") as file:
        influencers = [line.strip() for line in file.readlines()]

    size_with_duplicates: int = len(influencers)
    remove_duplicate_list = set(influencers)
    size_without_duplicates: int = len(remove_duplicate_list)

    if size_with_duplicates == size_without_duplicates:
        print("Already Up-to-date")
        return
    
    """with open("../insta_profiles.txt", "w") as file:
        for influencer in remove_duplicate_list:
            file.write(influencer + "\n")
    """
    print(f"Removed duplicates. {size_with_duplicates - size_without_duplicates} entries deleted.")


In [13]:
from load import Postgres
import psycopg2

In [14]:
conn = psycopg2.connect(database="postgres", user="postgres", password=1040)
cur = conn.cursor()

In [15]:
def does_mention_exist(username: str) -> bool:
    query = """
        SELECT p.caption
        FROM posts p
        JOIN influencers i
        ON p.ownerid = i.influencerid
        WHERE i.username = %s
    """
    cur = conn.cursor()
    cur.execute(query, (username,))
    result = cur.fetchall()
    cur.close()
    
    return result

In [16]:
res = does_mention_exist("mahirahkhan")
res[47]

('Little bit of Love Guru bts x \n\nP.S I don’t know how to jog and I begged my director to show me walking or something but he wanted a JOG and ugh it was tough. 🫣\n\nAlso notice how I had toot Gaya playing through every scene… whattta whattaa song! Can’t wait for it to release/ inshAllah.',)

In [164]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

url1="https://starngage.com/plus/en/brand/ranking/instagram/pakistan/politics"

url="https://starngage.com/plus/en/influencer/ranking/instagram/pakistan"
driver = webdriver.Chrome()  # or webdriver.Firefox()
driver.get(url)

# Wait for the table to load
wait = WebDriverWait(driver, 10)
tbody = wait.until(EC.presence_of_element_located((By.TAG_NAME, "tbody")))

# Find all name links
name_links = driver.find_elements(By.CSS_SELECTOR, "tbody tr .name a")
names = [link.text for link in name_links if link.text.strip()]

driver.quit()

cleaned_names = [name.lstrip('@') for name in names]

print(len(cleaned_names))

with open("../insta_profiles.txt", "a") as file:
    for username in cleaned_names:
        file.write(username + "\n")

100
